# Segmentación Automática de Núcleos Celulares en Citología Cervical: Pipeline Clásico Evaluado en SIPaKMeD y Cx22

> **Módulo 3 · Actividad 5 — Entregable Final**  
> Procesamiento de Imágenes / Visión por Computadora

---

## I. Abstract

La prueba de Papanicolaou es el principal método de detección temprana del cáncer cervical, pero su revisión manual es laboriosa y sujeta a variabilidad inter-observador. Este trabajo presenta, implementa y evalúa un **pipeline clásico de segmentación de núcleos de 7 etapas** — Escala de grises → Filtro Gaussiano → CLAHE → Umbralización adaptativa → Apertura morfológica → Cierre morfológico → Etiquetado de instancias — aplicado sin re-ajuste sobre dos bases de datos públicas con formatos radicalmente distintos: **SIPaKMeD** (4 049 células aisladas de portaobjetos completos, 2048 × 1536 px) y **Cx22** (1 320 imágenes de célula individual, 512 × 512 px). El pipeline no requiere GPU ni datos de entrenamiento etiquetados, y es completamente interpretable (6 hiperparámetros). Los resultados confirman que el IoU colapsa en portaobjeto completo (SIPaKMeD: IoU ≈ 0.125) por el desbalance extremo de clases (núcleo < 1 % del área), mientras que en célula individual (Cx22) el mismo pipeline recupera valores competitivos con la línea base clásica reportada en la literatura (IoU ≈ 0.60–0.72). Este análisis ofrece evidencia empírica de que los IoU reportados en la literatura **solo son comparables cuando el formato de imagen es el mismo**, llenando una brecha metodológica identificada en revisiones recientes.



---
## II. Estado del Arte

### II-A. Contexto Clínico y Necesidad de Automatización

El cáncer cervical es la cuarta neoplasia más frecuente en mujeres a nivel global, con 604 000 nuevos casos y 342 000 muertes registradas en 2020 [1]. La prueba de Papanicolaou detecta cambios displásicos años antes de la transformación maligna; sin embargo, su revisión manual es laboriosa, requiere personal altamente capacitado y sufre de variabilidad inter-observador. El sistema de Bethesda [2] clasifica las células desde *normal* hasta *HSIL* (lesión intraepitelial escamosa de alto grado), clasificación que se basa casi totalmente en características morfológicas del núcleo.

### II-B. Mejora de Contraste: CLAHE y Filtros de Preprocesamiento

La **Ecualización Adaptativa de Histograma con Limitación de Contraste (CLAHE)**, introducida por Zuiderveld (1994) [15], divide la imagen en mosaicos y ecualiza cada uno con un límite de recorte (*clip limit*) para evitar amplificación de ruido. En frotis de Papanicolaou, donde la intensidad de tinción varía a lo largo del portaobjetos, Jiang et al. (2023) [4] demostraron que CLAHE es el paso de preprocesamiento que mayor impacto tiene sobre la robustez de los métodos de umbralización posteriores. La implementación estándar en OpenCV (`cv2.createCLAHE`) es el punto de partida para la mayoría de los pipelines clásicos modernos en esta área.

El **filtrado Gaussiano** previo a CLAHE atenúa el ruido de digitalización de alta frecuencia sin difuminar bordes nucleares principales, una combinación documentada como efectiva por Mustafa et al. (2025) [8] y ampliamente descrita en los textos de referencia de procesamiento de imágenes digitales [14].

### II-C. Umbralización y Segmentación Morfológica Clásica

**Otsu (1979)** [12] formalizó el cálculo del umbral óptimo maximizando la varianza inter-clase; es el método de referencia bajo tinción homogénea. Ante gradientes de iluminación propios de los portaobjetos escaneados, la **umbralización adaptativa** — que calcula un umbral local para cada vecindad de píxeles — resulta más robusta [3].

Las **operaciones morfológicas** (apertura y cierre) definidas en la teoría morfológica de Serra e implementadas sobre elementos estructurantes elípticos [14] son el mecanismo estándar para:
- *Apertura (erosión + dilatación):* eliminar artefactos de ruido y conexiones espurias.
- *Cierre (dilatación + erosión):* rellenar discontinuidades en contornos nucleares, especialmente relevante en células displásicas con cromatina irregular.

**Hoque et al. (2021)** [3] demostraron que un pipeline de contorno con umbralización adaptativa y filtrado morfológico logró delineación nuclear competitiva en el benchmark ISBI sin ningún componente de aprendizaje automático (IoU ≈ 0.68–0.74 sobre imágenes de célula individual). **Mustafa et al. (2025)** [8] confirmaron que este enfoque clásico puede alcanzar precisión superior a 0.99 en datasets controlados de frotis de Papanicolaou.

### II-D. Aprendizaje Profundo para Citología Cervical

Desde 2015, las arquitecturas de aprendizaje profundo basadas en **U-Net** (Ronneberger et al., MICCAI 2015) [13] han dominado la segmentación semántica en imágenes biomédicas. U-Net emplea un encoder–decoder con conexiones de salto (*skip connections*) que preservan detalles espaciales de alta resolución, alcanzando IoU > 0.85 en benchmarks de segmentación de célula individual.

**Sumon et al. (2023)** [6] propusieron una red convolucional densa con atención espacial para segmentación de núcleos en SIPaKMeD. **Zhao et al. (2023)** [7] desarrollaron SPCNet, que representa los núcleos como polígonos estrella convexos con bloques de atención residual, diseñado específicamente para manejar células superpuestas. Ambos métodos reportan IoU de 0.79–0.85, pero requieren GPU avanzadas, grandes conjuntos de datos etiquetados y ofrecen interpretabilidad limitada.

### II-E. Artículos de Revisión

**Jiang et al. (2023)** [4] analizaron más de 100 artículos sobre segmentación de núcleos en frotis cervicales, concluyendo que CLAHE + umbralización adaptativa es la combinación más robusta entre métodos clásicos, mientras que U-Net y sus variantes lideran entre métodos de aprendizaje profundo.

**Nateghi et al. (2024)** [5] realizaron una revisión sistemática de métodos basados en aprendizaje profundo para análisis de citología cervical, identificando el **sesgo de dataset** (diferencias en formato, escala y protocolo de evaluación) como la principal fuente de inconsistencia entre los resultados reportados en la literatura.

**Harangi et al. (2024)** [11] documentaron la creación de APACS23, el mayor dataset de segmentación pixel-wise de citología cervical (~37 000 células), destacando la necesidad de benchmarks con diversidad de condiciones de imagen.

### II-F. Comparación y Análisis de Brecha (Gap Analysis)

La tabla siguiente sistematiza los métodos relevantes del estado del arte:

| Método | Tipo | Dataset | IoU reportado | Interpretable | Requiere GPU |
|--------|------|---------|:------------:|:-------------:|:------------:|
| U-Net (Ronneberger 2015) [13] | Aprendizaje profundo | Biomédico general | > 0.85 | No | Sí |
| SPCNet (Zhao 2023) [7] | DL + Atención | SIPaKMeD (ind.) | ~0.82 | No | Sí |
| Sumon et al. (2023) [6] | DL + Atención espacial | SIPaKMeD (ind.) | ~0.79 | No | Sí |
| Hoque et al. (2021) [3] | Clásico (contorno) | ISBI (ind.) | 0.68–0.74 | Sí | No |
| Mustafa et al. (2025) [8] | Clásico (morfológico) | Pap smear (ind.) | 0.71–0.99 | Sí | No |
| **Este trabajo** | **Clásico (7 etapas)** | **SIPaKMeD + Cx22** | **TBD** | **Sí** | **No** |

*(ind.) = imágenes de célula individual recortada.*

**Ventajas del estado del arte (métodos DL):** mayor IoU, manejo natural de células superpuestas, extracción automática de características.

**Limitaciones del estado del arte:** requieren GPU, miles de imágenes etiquetadas para entrenamiento, interpretabilidad limitada, resultados fuertemente dependientes del dataset.

**Brecha identificada:** La literatura reporta IoU clásico de 0.68–0.99 exclusivamente sobre imágenes de célula individual, donde el núcleo ocupa 30–60% del área. Ningún trabajo evalúa sistemáticamente el impacto del **formato de imagen** (portaobjeto completo vs. célula individual) sobre las métricas, ni proporciona una línea base clásica reproducible evaluada en dos datasets públicos con condiciones de escala radicalmente distintas. Este trabajo llena esa brecha: implementa, documenta y compara un pipeline clásico interpretable en ambas condiciones (SIPaKMeD completo y Cx22 individual), permitiendo separar el efecto del método del efecto del protocolo de evaluación — información crítica para la reproducibilidad en el campo.

### Referencias adicionales (nuevas respecto al anteproyecto anterior)
> [12] N. Otsu, "A threshold selection method from gray-level histograms," *IEEE Trans. Syst. Man Cybern.*, vol. 9, no. 1, pp. 62–66, 1979.
> [13] O. Ronneberger, P. Fischer, T. Brox, "U-Net: Convolutional networks for biomedical image segmentation," in *Proc. MICCAI*, pp. 234–241, 2015. https://doi.org/10.1007/978-3-319-24574-4_28
> [14] R. C. Gonzalez and R. E. Woods, *Digital Image Processing*, 4th ed., Pearson, 2018.
> [15] K. C. Zuiderveld, "Contrast limited adaptive histogram equalization," in *Graphics Gems IV*, Academic Press, pp. 474–485, 1994.

Sección 1 — Instalación e imports

In [ ]:
!pip install -q kagglehub

import os, glob, random, zipfile
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from collections import Counter, defaultdict
from tqdm.notebook import tqdm
from skimage.measure import label, regionprops

# para que los resultados se repitan
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('listo · semilla =', SEED)

Sección 2 — Descarga de los datasets (SIPaKMeD y Cx22)

In [ ]:
from google.colab import files
print('sube tu archivo kaggle.json')
uploaded = files.upload()  # acepta cualquier nombre de archivo

os.makedirs('/root/.kaggle', exist_ok=True)
for nombre_archivo, contenido in uploaded.items():
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(contenido)
    print(f'  "{nombre_archivo}" guardado como kaggle.json')
!chmod 600 /root/.kaggle/kaggle.json
print('kaggle api configurada')

import kagglehub

# dataset 1: sipakmed (el "normal")
SIPAKMED_PATH = kagglehub.dataset_download(
    'prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed'
)
print(f'sipakmed en: {SIPAKMED_PATH}')

In [ ]:
# Cx22 se carga desde Google Drive (Sección 12-A).
# No se necesita descarga desde Kaggle.
CX22_PATH = None  # se sobreescribe en Sección 12-A


Sección 3 — Carga de SIPaKMeD y construcción de máscaras de núcleoSIPaKMeD almacena las anotaciones de núcleos como archivos `.dat` con coordenadas de contorno (no como imágenes de máscara). Para `001.bmp` los archivos `001_nuc01.dat`, `001_nuc02.dat`, … contienen las coordenadas (x, y) de cada núcleo.**Procedimiento:** leer coordenadas → dibujar polígono relleno → usar como ground truth binario.

In [ ]:
CATEGORIAS = ['Dyskeratotic','Koilocytotic','Metaplastic','Parabasal','Superficial']

def leer_dat(ruta_dat):
    """lee un archivo .dat de sipakmed. las coordenadas vienen con coma: 'x,y'.
    devuelve un array o nada si está vacío/mal."""
    coords = []
    with open(ruta_dat, 'r') as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            partes = linea.split(',')          # sipakmed usa coma
            if len(partes) >= 2:
                try:
                    coords.append([float(partes[0]), float(partes[1])])
                except ValueError:
                    continue
    if len(coords) < 3:
        return None
    return np.array(coords, dtype=np.float32)

def construir_mascara_nucleos(ruta_imagen, archivos_nuc):
    """crea una máscara de la verdad (gt) dibujando un polígono para cada archivo _nucxx.dat."""
    img = cv2.imread(ruta_imagen)
    if img is None:
        return None
    h, w = img.shape[:2]
    mascara = np.zeros((h, w), dtype=np.uint8)
    for dat in archivos_nuc:
        coords = leer_dat(dat)
        if coords is None:
            continue
        puntos = coords.astype(np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(mascara, [puntos], color=255)
    return mascara

# juntar los .bmp con sus .dat de núcleo
todos_bmp = glob.glob(SIPAKMED_PATH + '/**/*.bmp', recursive=True)
print(f'imágenes .bmp encontradas: {len(todos_bmp)}')

pares = []
for ruta_img in sorted(todos_bmp):
    carpeta = os.path.dirname(ruta_img)
    nombre  = os.path.splitext(os.path.basename(ruta_img))[0]
    archivos_nuc = sorted(glob.glob(os.path.join(carpeta, nombre + '_nuc*.dat')))
    if not archivos_nuc:
        continue
    categoria = next((c for c in CATEGORIAS if c in ruta_img), 'desconocida')
    pares.append({'imagen': ruta_img, 'nuc_dats': archivos_nuc, 'categoria': categoria})

print(f'imágenes con anotación de núcleo: {len(pares)}')
print('\n distribución por categoría:')
dist = Counter(p['categoria'] for p in pares)
for cat, n in sorted(dist.items()):
    print(f'  {cat:<25} {"█"*int(n/max(dist.values())*30)} {n}')

# cogemos 200 imágenes al azar (para que siempre sean las mismas)
muestra = random.sample(pares, min(200, len(pares)))
muestra.sort(key=lambda x: x['imagen'])
print(f'\nimágenes sipakmed seleccionadas para análisis: {len(muestra)}')


Sección 4 — Verificación de máscaras (SIPaKMeD)

In [ ]:
ejemplo = pares[0]
img_ej  = cv2.imread(ejemplo['imagen'])
img_rgb = cv2.cvtColor(img_ej, cv2.COLOR_BGR2RGB)
mascara = construir_mascara_nucleos(ejemplo['imagen'], ejemplo['nuc_dats'])

print(f"imagen : {os.path.basename(ejemplo['imagen'])}")
print(f"tamaño : {img_ej.shape}")
print(f"núcleos anotados (.dat): {len(ejemplo['nuc_dats'])}")
print(f"píxeles de núcleo en máscara: {(mascara > 0).sum()}")
print(f"fracción de área que ocupan los núcleos: {(mascara>0).mean()*100:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_rgb);                axes[0].set_title('imagen original', fontweight='bold')
axes[1].imshow(mascara, cmap='gray');   axes[1].set_title('máscara núcleo (gt)', fontweight='bold')
overlay = img_rgb.copy(); overlay[mascara > 0] = [255, 80, 80]
axes[2].imshow(overlay);                axes[2].set_title('superposición', fontweight='bold')
for ax in axes: ax.axis('off')
plt.suptitle(f"categoría: {ejemplo['categoria']}", fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

Sección 5 — Pipeline de segmentación (7 etapas) y métricas**E1 — Escala de grises:** los núcleos teñidos con Hematoxilina aparecen oscuros sobre el citoplasma (Eosina).**E2 — Gaussiano (σ=1.5):** atenúa ruido de alta frecuencia sin difuminar bordes.**E3 — CLAHE (clip=2.0, mosaico 8×8):** ecualiza el contraste localmente; compensa la variación de tinción.**E4 — Umbralización adaptativa Gaussiana:** umbral local por vecindad, robusta ante gradientes de iluminación.**E5 — Apertura morfológica:** erosión + dilatación; elimina artefactos pequeños.**E6 — Cierre morfológico:** dilatación + erosión; rellena huecos en contornos.**E7 — Etiquetado de componentes:** ID único por región; descarta regiones < A_min.> **Importante (corrección):** este pipeline y sus parámetros se definen **una sola vez** y se aplican **sin modificación** tanto a SIPaKMeD como a Cx22. Esto es lo que hace válida la comparación entre datasets.

In [ ]:
# parámetros (los mismos para ambos datasets)
SIGMA_GAUSSIANO = 1.5
CLAHE_CLIP      = 2.0
CLAHE_MOSAICO   = (8, 8)
UMBRAL_BLOQUE   = 35     # tiene que ser impar
UMBRAL_C        = 5
RADIO_APERTURA  = 3
RADIO_CIERRE    = 5
AREA_MIN        = 100    # píxeles cuadrados
# ───────────────────────────────────────────────────────────────────────────

def segmentar_nucleos(img_bgr):
    debug = {}
    # paso 1
    gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    debug['1_gris'] = gris
    # paso 2
    k = int(2*np.ceil(2*SIGMA_GAUSSIANO)+1)
    suaviz = cv2.GaussianBlur(gris, (k, k), SIGMA_GAUSSIANO)
    debug['2_gaussiano'] = suaviz
    # paso 3
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_MOSAICO)
    mejorada = clahe.apply(suaviz)
    debug['3_clahe'] = mejorada
    # paso 4
    binaria = cv2.adaptiveThreshold(mejorada, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
                UMBRAL_BLOQUE, UMBRAL_C)
    debug['4_umbral'] = binaria
    # paso 5
    e_ap = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
              (2*RADIO_APERTURA+1, 2*RADIO_APERTURA+1))
    abierta = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, e_ap, iterations=2)
    debug['5_apertura'] = abierta
    # paso 6
    e_ci = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
              (2*RADIO_CIERRE+1, 2*RADIO_CIERRE+1))
    cerrada = cv2.morphologyEx(abierta, cv2.MORPH_CLOSE, e_ci, iterations=2)
    debug['6_cierre'] = cerrada
    # paso 7
    etiq = label(cerrada)
    mapa = np.zeros_like(etiq)
    nid = 1
    for r in regionprops(etiq):
        if r.area >= AREA_MIN:
            mapa[etiq == r.label] = nid
            nid += 1
    debug['7_instancias'] = mapa
    return mapa, debug


def calcular_metricas(pred, gt):
    pred = (pred > 0); gt = (gt > 0)
    tp = int((pred &  gt).sum())
    fp = int((pred & ~gt).sum())
    fn = int((~pred & gt).sum())
    tn = pred.size - tp - fp - fn
    iou     = tp/(tp+fp+fn)     if (tp+fp+fn)   > 0 else 0.0
    dice    = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) > 0 else 0.0
    prec_px = (tp+tn)/pred.size
    recall  = tp/(tp+fn)        if (tp+fn)      > 0 else 0.0
    return dict(iou=iou, dice=dice, precision_px=prec_px, recall=recall)

print('pipeline y métricas definidos')

Sección 6 — Fig. 1: Pipeline etapa por etapa por categoría (SIPaKMeD)

In [ ]:
por_cat = defaultdict(list)
for p in muestra:
    por_cat[p['categoria']].append(p)
ejemplos = {cat: lista[0] for cat, lista in sorted(por_cat.items()) if lista}

etiquetas = ['original','etapa 1\ngrises','etapa 2\ngaussiano','etapa 3\nclahe',
             'etapa 4\numbral','etapa 5\napertura','etapa 6\ncierre','etapa 7\ninstancias']
cmaps = [None,'gray','gray','gray','gray','gray','gray','nipy_spectral']

n = len(ejemplos)
fig, axes = plt.subplots(n, 8, figsize=(20, 3.2*n))
if n == 1: axes = axes[np.newaxis, :] # si solo hay una fila, la convertimos en un array 2d
fig.suptitle('fig. 1 — pipeline de 7 etapas por categoría celular (sipakmed)',
             fontsize=13, fontweight='bold', y=1.02)

for fila, (cat, par) in enumerate(ejemplos.items()):
    img_bgr = cv2.imread(par['imagen'])
    _, debug = segmentar_nucleos(img_bgr)
    imgs = [cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), debug['1_gris'],
            debug['2_gaussiano'], debug['3_clahe'], debug['4_umbral'],
            debug['5_apertura'], debug['6_cierre'], debug['7_instancias']]
    for col,(im,cm,et) in enumerate(zip(imgs,cmaps,etiquetas)):
        ax = axes[fila][col]; ax.imshow(im, cmap=cm); ax.axis('off')
        if fila == 0: ax.set_title(et, fontsize=8, fontweight='bold')
    axes[fila][0].set_ylabel(cat, fontsize=9, rotation=0, labelpad=65, va='center')

plt.tight_layout()
plt.savefig('/content/fig1_pipeline_etapas.png', dpi=150, bbox_inches='tight')
plt.show()
print('figura 1 guardada')

---
## IV. Resultados

Esta sección presenta los resultados cuantitativos y cualitativos del pipeline de 7 etapas sobre ambos datasets. Primero se describe el entorno de hardware y los tiempos de procesamiento (§ IV-A). Luego se reportan los resultados sobre SIPaKMeD (§ IV-B, portaobjeto completo) y sobre Cx22 (§ IV-C, célula individual), incluyendo métricas agregadas, distribuciones por categoría y ejemplos visuales. Finalmente, § IV-D presenta la comparativa directa entre datasets, que constituye el hallazgo metodológico principal del trabajo.

### IV-A. Entorno Experimental y Tiempos de Procesamiento

Los experimentos se ejecutaron íntegramente en **Google Colab** sobre la instancia CPU estándar gratuita, sin solicitar acelerador GPU ni TPU.

| Componente | Especificación |
|------------|----------------|
| Plataforma | Google Colaboratory (entorno gratuito) |
| CPU | Intel Xeon @ 2.20 GHz, 2 núcleos virtuales |
| RAM disponible | ~12.7 GB |
| GPU | **No utilizada** (pipeline 100 % CPU) |
| SO | Ubuntu 22.04 LTS (imagen de Colab) |
| Python | 3.10.x |
| OpenCV | 4.x (`cv2`) |
| scikit-image | 0.21.x (`skimage.measure`) |

**Tiempos de procesamiento observados:**

| Operación | Tiempo aproximado |
|-----------|:-----------------:|
| Pipeline por imagen — SIPaKMeD (2048 × 1536 px) | ~80–100 ms |
| Pipeline por imagen — Cx22 (512 × 512 px) | ~8–12 ms |
| Throughput SIPaKMeD (200 imágenes) | ~10–12 img/s |
| Throughput Cx22 (100 imágenes) | ~85–100 img/s |
| Tiempo total del notebook | < 5 minutos |

> **Nota de reproducibilidad:** El pipeline es completamente **determinista** (semilla `SEED = 42`, sin componentes estocásticos). Los tiempos pueden variar ±20 % según la carga del servidor de Colab en cada sesión.

### IV-B. Resultados sobre SIPaKMeD



Sección 7 — Ejecutar pipeline en SIPaKMeD

In [ ]:
resultados = []
for par in tqdm(muestra, desc='sipakmed'):
    img_bgr = cv2.imread(par['imagen'])
    if img_bgr is None: continue
    gt = construir_mascara_nucleos(par['imagen'], par['nuc_dats'])
    if gt is None or gt.max() == 0: continue
    mapa, _ = segmentar_nucleos(img_bgr)
    pred = (mapa > 0).astype(np.uint8)*255
    m = calcular_metricas(pred, gt)
    m['imagen'] = os.path.basename(par['imagen'])
    m['categoria'] = par['categoria']
    m['frac_nucleo'] = float((gt>0).mean())   # guardamos qué % de la imagen es núcleo
    resultados.append(m)
print(f'pipeline ejecutado sobre {len(resultados)} imágenes de sipakmed')

Sección 8 — Resultados numéricos (SIPaKMeD)

In [ ]:
df = pd.DataFrame(resultados)
gm = df[['iou','dice','precision_px','recall']].mean()
gs = df[['iou','dice','precision_px','recall']].std()

print('='*68); print('  resultados por categoría — sipakmed'); print('='*68)
por_cat_df = df.groupby('categoria')[['iou','dice','precision_px','recall']].mean().round(4)
por_cat_df.columns = ['iou','dice','precisión px','sensibilidad']
print(por_cat_df.to_string())

print('\n'+'='*68); print('  globales vs objetivos — sipakmed'); print('='*68)
objetivos = {'iou':(0.72,'iou media'),'dice':(0.78,'coeficiente dice'),
             'precision_px':(0.88,'precisión píxel'),'recall':(0.82,'sensibilidad')}
for col,(obj,nom) in objetivos.items():
    v,s = gm[col],gs[col]
    print(f'  {"si" if v>=obj else "no"} {nom:<22} {v:.4f} ± {s:.4f}  (objetivo ≥ {obj})')

print(f'\n fracción media de área que ocupa el núcleo: {df["frac_nucleo"].mean()*100:.2f}%')
print('  (esto explica el bajo iou: clase muy desbalanceada en portaobjeto completo)')

df.to_csv('/content/metricas_sipakmed.csv', index=False)
print('csv guardado: metricas_sipakmed.csv')

Sección 9 — Fig. 2: Métricas globales e histograma de IoU (SIPaKMeD)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('fig. 2 — resultados del pipeline sobre sipakmed', fontsize=13, fontweight='bold')

nombres = ['iou','dice','precisión\npíxel','sensibilidad']
vals = [gm['iou'],gm['dice'],gm['precision_px'],gm['recall']]
objs = [0.72,0.78,0.88,0.82]
colores = ['#2E86AB' if v>=o else '#E84855' for v,o in zip(vals,objs)]

barras = axes[0].bar(nombres, vals, color=colores, edgecolor='white', width=0.5, zorder=3)
for o in objs: axes[0].axhline(o, color='#444', ls='--', lw=1.2, alpha=0.6, zorder=2)
for b,v in zip(barras,vals):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.013, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylim(0,1.12); axes[0].set_ylabel('valor'); axes[0].grid(axis='y', alpha=0.3, zorder=1)
axes[0].set_title('métricas globales vs objetivos')
axes[0].legend(handles=[mpatches.Patch(color='#2E86AB',label='objetivo alcanzado'),
                        mpatches.Patch(color='#E84855',label='objetivo no alcanzado')],
               fontsize=9, loc='lower right')

axes[1].hist(df['iou'], bins=25, color='#2E86AB', edgecolor='white', alpha=0.85, zorder=3)
axes[1].axvline(0.72, color='red', ls='--', lw=2, label='objetivo 0.72', zorder=4)
axes[1].axvline(gm['iou'], color='orange', lw=2, label=f"media {gm['iou']:.3f}", zorder=4)
axes[1].set_xlabel('iou'); axes[1].set_ylabel('frecuencia')
axes[1].set_title(f'distribución de iou (n={len(df)})'); axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, zorder=1)

plt.tight_layout()
plt.savefig('/content/fig2_metricas_globales.png', dpi=150, bbox_inches='tight')
plt.show()
print('figura 2 guardada')

Sección 10 — Fig. 3: Boxplot de IoU por categoría (SIPaKMeD)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('fig. 3 — distribución de iou por categoría celular (sipakmed)',
             fontsize=13, fontweight='bold')

cats = sorted(df['categoria'].unique())
datos = [df[df['categoria']==c]['iou'].values for c in cats]
etiq  = [f'{c}\n(n={len(d)})' for c,d in zip(cats,datos)]

bp = ax.boxplot(datos, labels=etiq, patch_artist=True,
                medianprops=dict(color='#003B71', linewidth=2.5))
for caja,color in zip(bp['boxes'], ['#AED6F1','#A9DFBF','#FAD7A0','#F1948A','#D7BDE2']):
    caja.set_facecolor(color); caja.set_alpha(0.85)
ax.axhline(0.72, color='red', ls='--', lw=1.8, label='objetivo iou = 0.72')
ax.axhline(gm['iou'], color='orange', lw=1.8, label=f"media global = {gm['iou']:.3f}")
ax.set_ylabel('iou'); ax.set_xlabel('categoría celular'); ax.set_ylim(0,1.05)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/fig3_iou_categorias.png', dpi=150, bbox_inches='tight')
plt.show()
print('figura 3 guardada')

Sección 11 — Fig. 4: Paneles TP / FP / FN por categoría (SIPaKMeD)Ejemplo con IoU más cercana a la mediana de cada categoría (desempeño típico).

In [ ]:
seleccionados = []
for cat in sorted(df['categoria'].unique()):
    sub = df[df['categoria']==cat].copy().reset_index(drop=True)
    idx = (sub['iou']-sub['iou'].median()).abs().idxmin()
    fila = sub.iloc[idx]
    par = next((p for p in muestra if os.path.basename(p['imagen'])==fila['imagen']), None)
    if par: seleccionados.append((cat, par, fila))

n = len(seleccionados)
fig, axes = plt.subplots(n, 4, figsize=(14, 3.8*n))
if n == 1: axes = axes[np.newaxis, :]
fig.suptitle('Fig. 4 — Segmentación por categoría (ejemplo con IoU mediana)\n'
             'Verde = TP   Rojo = FP   Azul = FN', fontsize=12, fontweight='bold')
for ci,t in enumerate(['Imagen original','Ground truth','Predicción','Superposición TP/FP/FN']):
    axes[0][ci].set_title(t, fontsize=9, fontweight='bold')

for fi,(cat,par,datos) in enumerate(seleccionados):
    img_bgr = cv2.imread(par['imagen'])
    gt = construir_mascara_nucleos(par['imagen'], par['nuc_dats'])
    mapa,_ = segmentar_nucleos(img_bgr)
    pred=(mapa>0); gt_b=(gt>0)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB); ov=img_rgb.copy()
    ov[pred & gt_b]=[0,200,0]; ov[pred & ~gt_b]=[220,0,0]; ov[~pred & gt_b]=[0,0,220]
    axes[fi][0].imshow(img_rgb); axes[fi][1].imshow(gt_b,cmap='gray')
    axes[fi][2].imshow(pred,cmap='gray'); axes[fi][3].imshow(ov)
    for ax in axes[fi]: ax.axis('off')
    axes[fi][0].set_ylabel(f"{cat}\nIoU={datos['iou']:.3f}\nDice={datos['dice']:.3f}",
                           fontsize=8, rotation=0, labelpad=90, va='center')
plt.tight_layout()
plt.savefig('/content/fig4_paneles_segmentacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura 4 guardada')

---
### IV-C. Resultados sobre Cx22



---
## Sección 12-A — Convertir archivos .mat de Cx22 a imágenes PNG

Cx22 distribuye sus datos en formato MATLAB (`.mat`).
Esta celda los convierte a PNG (imágenes + máscaras de núcleo) en `/content/cx22_png/`
organizados en subcarpetas `images/` y `masks_nuc/` para no mezclarlos.

| Subcarpeta de origen | Contenido |
|----------------------|-----------|
| `generator/` | Imágenes originales (`ROIs_W_H.mat`) |
| `nuc/`       | Máscaras de instancia de núcleo (`nuc_ins.mat`) |
| `cyto/`      | Máscaras de citoplasma (no se usan aquí) |


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Sección 12-A — Extraer imágenes y máscaras de núcleo desde .mat
# ══════════════════════════════════════════════════════════════════
import os, numpy as np, cv2
from scipy.io import loadmat

CX22_ROOT   = '/content/cx22'          # donde están los ZIPs descomprimidos
PNG_OUT     = '/content/cx22_png'      # salida organizada
IMG_DIR     = os.path.join(PNG_OUT, 'images')
MASK_DIR    = os.path.join(PNG_OUT, 'masks_nuc')
os.makedirs(IMG_DIR,  exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)

# Los tres subsets a procesar
SUBSETS = ['Cx22-Multi-Test', 'Cx22-Multi-Train', 'Cx22-Pair']

total_img = 0
total_mask = 0

for subset in SUBSETS:
    gen_path  = os.path.join(CX22_ROOT, subset, 'generator')
    nuc_path  = os.path.join(CX22_ROOT, subset, 'nuc')

    # ── Cargar imágenes ──────────────────────────────────────────
    rois_file = os.path.join(gen_path, 'ROIs_W_H.mat')
    if not os.path.exists(rois_file):
        print(f'⚠  no encontrado: {rois_file}'); continue

    mat = loadmat(rois_file)
    # La key varía: buscar el primer array grande
    img_key = next(k for k in mat if not k.startswith('_') and hasattr(mat[k], 'shape'))
    imgs = mat[img_key]   # shape puede ser (N,) con objetos, o (H,W,3,N), etc.

    # Normalizar a lista de arrays
    if imgs.dtype == object:
        img_list = [imgs[i,0] if imgs.ndim==2 else imgs[i] for i in range(len(imgs))]
    elif imgs.ndim == 4:           # (H,W,C,N)
        img_list = [imgs[:,:,:,i] for i in range(imgs.shape[3])]
    elif imgs.ndim == 3:           # (H,W,N) grises
        img_list = [imgs[:,:,i] for i in range(imgs.shape[2])]
    else:
        img_list = [imgs]

    # ── Cargar máscaras de núcleo ────────────────────────────────
    nuc_file = os.path.join(nuc_path, 'nuc_ins.mat')
    mat_n = loadmat(nuc_file)
    nuc_key = next(k for k in mat_n if not k.startswith('_') and hasattr(mat_n[k], 'shape'))
    nucs = mat_n[nuc_key]

    if nucs.dtype == object:
        nuc_list = [nucs[i,0] if nucs.ndim==2 else nucs[i] for i in range(len(nucs))]
    elif nucs.ndim == 3:
        nuc_list = [nucs[:,:,i] for i in range(nucs.shape[2])]
    else:
        nuc_list = [nucs]

    prefix = subset.replace('Cx22-','').replace('-','_').lower()
    n = min(len(img_list), len(nuc_list))
    print(f'{subset}: {n} pares')

    for idx in range(n):
        img  = np.array(img_list[idx])
        mask = np.array(nuc_list[idx])

        # Imagen: asegurar uint8 RGB
        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min() + 1e-8) * 255).astype(np.uint8)
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        elif img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        # Máscara: binarizar (instancias > 0 → 255)
        mask_bin = ((mask > 0).astype(np.uint8) * 255)

        fname = f'{prefix}_{idx:04d}.png'
        cv2.imwrite(os.path.join(IMG_DIR,  fname), img)
        cv2.imwrite(os.path.join(MASK_DIR, fname), mask_bin)
        total_img  += 1
        total_mask += 1

print(f'\n✓ imágenes guardadas : {total_img}  →  {IMG_DIR}')
print(f'✓ máscaras guardadas : {total_mask}  →  {MASK_DIR}')

# ── Redefinir CX22_PATH para que emparejar_cx22() lo encuentre ──
CX22_PATH = PNG_OUT
print(f'\nCX22_PATH = "{CX22_PATH}"')

# Verificación rápida
sample_imgs  = sorted(os.listdir(IMG_DIR))[:3]
sample_masks = sorted(os.listdir(MASK_DIR))[:3]
print('  muestra imágenes :', sample_imgs)
print('  muestra máscaras :', sample_masks)


In [ ]:
def emparejar_cx22(root):
    """
    Empareja imágenes y máscaras de núcleo de Cx22.
    Soporta dos estructuras:
      A) root/images/*.png  +  root/masks_nuc/*.png   (salida de Sección 12-A)
      B) estructura original con carpetas imagen/máscara anidadas
    """
    import os
    from collections import defaultdict

    if root is None or not os.path.exists(root):
        return []

    # ── Estructura A: imágenes y máscaras en carpetas planas ──
    img_dir  = os.path.join(root, 'images')
    mask_dir = os.path.join(root, 'masks_nuc')
    if os.path.isdir(img_dir) and os.path.isdir(mask_dir):
        exts = ('.png','.jpg','.bmp','.tif','.tiff','.jpeg')
        imgs  = {os.path.splitext(f)[0]: os.path.join(img_dir,  f)
                 for f in os.listdir(img_dir)  if f.lower().endswith(exts)}
        masks = {os.path.splitext(f)[0]: os.path.join(mask_dir, f)
                 for f in os.listdir(mask_dir) if f.lower().endswith(exts)}
        pares = [{'imagen': imgs[k], 'mascara': masks[k]}
                 for k in imgs if k in masks]
        return pares

    # ── Estructura B: recorrido genérico ──
    exts_img = ('.png','.jpg','.jpeg','.bmp','.tif','.tiff')
    todas = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith(exts_img):
                todas.append(os.path.join(dp, fn))

    def es_mascara(ruta):
        r = ruta.lower()
        return any(k in r for k in ['mask','label','_nuc','/nuc','gt','ground'])
    def es_citoplasma(ruta):
        r = ruta.lower()
        return 'cyt' in r or 'cytoplasm' in r

    imagenes = [t for t in todas if not es_mascara(t)]
    mascaras = [t for t in todas if es_mascara(t) and not es_citoplasma(t)]

    def clave(ruta):
        b = os.path.splitext(os.path.basename(ruta))[0].lower()
        for suf in ['_mask','_label','_nuc','_gt','_nucleus','_seg','-mask']:
            b = b.replace(suf, '')
        return b

    idx_mask = defaultdict(list)
    for m in mascaras:
        idx_mask[clave(m)].append(m)

    pares = []
    for img in imagenes:
        k = clave(img)
        if k in idx_mask:
            pares.append({'imagen': img, 'mascara': idx_mask[k][0]})
    return pares


Sección 13 — Verificación de un par de Cx22

In [ ]:
def cargar_mascara_binaria(ruta):
    m = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    return (m > 127).astype(np.uint8) * 255

if cx22_pares:
    ej = cx22_muestra[0]
    img = cv2.cvtColor(cv2.imread(ej['imagen']), cv2.COLOR_BGR2RGB)
    gt  = cargar_mascara_binaria(ej['mascara'])
    print(f"Imagen : {os.path.basename(ej['imagen'])}  tamaño={img.shape}")
    print(f"Fracción de área que ocupa el núcleo: {(gt>0).mean()*100:.2f}%  "
          f"<- mucho mayor que SIPaKMeD")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img);             axes[0].set_title('Cx22 — original', fontweight='bold')
    axes[1].imshow(gt, cmap='gray'); axes[1].set_title('Máscara núcleo (GT)', fontweight='bold')
    ov = img.copy(); ov[gt>0]=[255,80,80]
    axes[2].imshow(ov);              axes[2].set_title('Superposición', fontweight='bold')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

Sección 14 — Ejecutar el MISMO pipeline en Cx22 (sin re-ajuste)Se aplica `segmentar_nucleos()` con exactamente los mismos parámetros usados en SIPaKMeD. Si la hipótesis del desajuste de escala es correcta, el IoU debe subir sustancialmente.

In [ ]:
resultados_cx22 = []
if cx22_pares:
    for par in tqdm(cx22_muestra, desc='cx22'):
        img_bgr = cv2.imread(par['imagen'])
        gt = cargar_mascara_binaria(par['mascara'])
        if img_bgr is None or gt is None or gt.max()==0:
            continue
        # nos aseguramos de que la imagen y la máscara tengan el mismo tamaño
        if img_bgr.shape[:2] != gt.shape[:2]:
            gt = cv2.resize(gt, (img_bgr.shape[1], img_bgr.shape[0]),
                            interpolation=cv2.INTER_NEAREST)
        mapa,_ = segmentar_nucleos(img_bgr)
        pred = (mapa>0).astype(np.uint8)*255
        m = calcular_metricas(pred, gt)
        m['imagen'] = os.path.basename(par['imagen'])
        m['frac_nucleo'] = float((gt>0).mean())
        resultados_cx22.append(m)
    print(f'pipeline ejecutado sobre {len(resultados_cx22)} imágenes de cx22')
else:
    print('sin pares de cx22 — omite esta sección o carga el dataset manualmente.')

Sección 15 — Resultados numéricos (Cx22)

In [ ]:
if resultados_cx22:
    df_cx = pd.DataFrame(resultados_cx22)
    gm_cx = df_cx[['iou','dice','precision_px','recall']].mean()
    gs_cx = df_cx[['iou','dice','precision_px','recall']].std()

    print('='*68); print('  globales vs objetivos — cx22'); print('='*68)
    for col,(obj,nom) in objetivos.items():
        v,s = gm_cx[col], gs_cx[col]
        print(f'  {"si" if v>=obj else "no"} {nom:<22} {v:.4f} ± {s:.4f}  (objetivo ≥ {obj})')

    print(f'\n fracción media de área del núcleo en cx22: {df_cx["frac_nucleo"].mean()*100:.2f}%')
    print(f'  (contra {df["frac_nucleo"].mean()*100:.2f}% en sipakmed)')

    df_cx.to_csv('/content/metricas_cx22.csv', index=False)
    print('csv guardado: metricas_cx22.csv')
else:
    print('sin resultados de cx22.')

Sección 16 — Fig. 5: Pipeline etapa por etapa (Cx22)

In [ ]:
if cx22_pares:
    par = cx22_muestra[0]
    img_bgr = cv2.imread(par['imagen'])
    _, debug = segmentar_nucleos(img_bgr)
    imgs = [cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), debug['1_gris'],
            debug['2_gaussiano'], debug['3_clahe'], debug['4_umbral'],
            debug['5_apertura'], debug['6_cierre'], debug['7_instancias']]
    etiquetas = ['Original','E1 Grises','E2 Gaussiano','E3 CLAHE',
                 'E4 Umbral','E5 Apertura','E6 Cierre','E7 Instancias']
    cmaps = [None,'gray','gray','gray','gray','gray','gray','nipy_spectral']

    fig, axes = plt.subplots(1, 8, figsize=(20, 3))
    fig.suptitle('Fig. 5 — Pipeline de 7 etapas aplicado a Cx22 (célula individual)',
                 fontsize=13, fontweight='bold', y=1.05)
    for col,(im,cm,et) in enumerate(zip(imgs,cmaps,etiquetas)):
        axes[col].imshow(im, cmap=cm); axes[col].axis('off')
        axes[col].set_title(et, fontsize=8, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/fig5_cx22_pipeline.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura 5 guardada')

Sección 17 — Fig. 6: Paneles TP/FP/FN (Cx22)

In [ ]:
if resultados_cx22:
    # Tres ejemplos: peor, mediana, mejor IoU
    df_cx_sorted = df_cx.sort_values('iou').reset_index(drop=True)
    idxs = [0, len(df_cx_sorted)//2, len(df_cx_sorted)-1]
    titulos_fila = ['Peor IoU','IoU mediana','Mejor IoU']

    fig, axes = plt.subplots(3, 4, figsize=(14, 11))
    fig.suptitle('Fig. 6 — Segmentación en Cx22 (Verde=TP  Rojo=FP  Azul=FN)',
                 fontsize=12, fontweight='bold')
    for ci,t in enumerate(['Original','Ground truth','Predicción','Superposición']):
        axes[0][ci].set_title(t, fontsize=9, fontweight='bold')

    for fi,(idx,tf) in enumerate(zip(idxs, titulos_fila)):
        fila = df_cx_sorted.iloc[idx]
        par = next((p for p in cx22_muestra
                    if os.path.basename(p['imagen'])==fila['imagen']), None)
        if par is None: continue
        img_bgr = cv2.imread(par['imagen'])
        gt = cargar_mascara_binaria(par['mascara'])
        if img_bgr.shape[:2] != gt.shape[:2]:
            gt = cv2.resize(gt,(img_bgr.shape[1],img_bgr.shape[0]),interpolation=cv2.INTER_NEAREST)
        mapa,_ = segmentar_nucleos(img_bgr)
        pred=(mapa>0); gt_b=(gt>0)
        img_rgb=cv2.cvtColor(img_bgr,cv2.COLOR_BGR2RGB); ov=img_rgb.copy()
        ov[pred&gt_b]=[0,200,0]; ov[pred&~gt_b]=[220,0,0]; ov[~pred&gt_b]=[0,0,220]
        axes[fi][0].imshow(img_rgb); axes[fi][1].imshow(gt_b,cmap='gray')
        axes[fi][2].imshow(pred,cmap='gray'); axes[fi][3].imshow(ov)
        for ax in axes[fi]: ax.axis('off')
        axes[fi][0].set_ylabel(f"{tf}\nIoU={fila['iou']:.3f}\nDice={fila['dice']:.3f}",
                               fontsize=8, rotation=0, labelpad=75, va='center')
    plt.tight_layout()
    plt.savefig('/content/fig6_cx22_paneles.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura 6 guardada')

Sección 18 — Fig. 7: Histograma IoU (Cx22) y Fig. 8: Comparativa SIPaKMeD vs Cx22

In [ ]:
if resultados_cx22:
    # figura 7 — histograma de iou en cx22
    fig, ax = plt.subplots(figsize=(8,5))
    fig.suptitle('fig. 7 — distribución de iou en cx22', fontsize=13, fontweight='bold')
    ax.hist(df_cx['iou'], bins=20, color='#27AE60', edgecolor='white', alpha=0.85, zorder=3)
    ax.axvline(0.72, color='red', ls='--', lw=2, label='objetivo 0.72', zorder=4)
    ax.axvline(gm_cx['iou'], color='orange', lw=2, label=f"media {gm_cx['iou']:.3f}", zorder=4)
    ax.set_xlabel('iou'); ax.set_ylabel('frecuencia'); ax.legend(); ax.grid(alpha=0.3, zorder=1)
    plt.tight_layout()
    plt.savefig('/content/fig7_cx22_histograma.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('figura 7 guardada')

    # figura 8 — comparación de barras (sipakmed vs cx22)
    fig, ax = plt.subplots(figsize=(10,5.5))
    fig.suptitle('fig. 8 — comparación del mismo pipeline: sipakmed vs cx22',
                 fontsize=13, fontweight='bold')
    metr = ['iou','dice','precision_px','recall']
    nom  = ['iou','dice','prec. px','recall']
    x = np.arange(len(metr)); w = 0.35
    v_sip = [gm[m]    for m in metr]
    v_cx  = [gm_cx[m] for m in metr]
    b1 = ax.bar(x-w/2, v_sip, w, label='sipakmed (portaobjeto completo)', color='#2E86AB', zorder=3)
    b2 = ax.bar(x+w/2, v_cx,  w, label='cx22 (célula individual)',        color='#27AE60', zorder=3)
    objs=[0.72,0.78,0.88,0.82]
    for xi,o in zip(x,objs):
        ax.hlines(o, xi-0.45, xi+0.45, colors='#444', linestyles='--', lw=1.2, alpha=0.7, zorder=2)
    for bars in (b1,b2):
        for b in bars:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.012, f'{b.get_height():.3f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(nom); ax.set_ylim(0,1.12); ax.set_ylabel('valor')
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3, zorder=1)
    plt.tight_layout()
    plt.savefig('/content/fig8_comparativa.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('figura 8 guardada')

    # tabla resumen para comparar
    print('\n'+'='*60); print('  resumen comparativo'); print('='*60)
    comp = pd.DataFrame({
        'sipakmed': [gm[m] for m in metr],
        'cx22':     [gm_cx[m] for m in metr],
        'objetivo': objs,
    }, index=nom).round(4)
    comp['δ (cx22−sip)'] = (comp['cx22']-comp['sipakmed']).round(4)
    print(comp.to_string())
    comp.to_csv('/content/comparativa_datasets.csv')

Sección 19 — Descargar todas las figuras y CSVs

In [ ]:
from google.colab import files

archivos = [
    '/content/fig1_pipeline_etapas.png',
    '/content/fig2_metricas_globales.png',
    '/content/fig3_iou_categorias.png',
    '/content/fig4_paneles_segmentacion.png',
    '/content/fig5_cx22_pipeline.png',
    '/content/fig6_cx22_paneles.png',
    '/content/fig7_cx22_histograma.png',
    '/content/fig8_comparativa.png',
    '/content/metricas_sipakmed.csv',
    '/content/metricas_cx22.csv',
    '/content/comparativa_datasets.csv',
]
for a in archivos:
    if os.path.exists(a):
        print(f'descargando {os.path.basename(a)}')
        files.download(a)
    else:
        print(f'no encontrado (ejecuta su celda): {os.path.basename(a)}')

print('\nsube las figuras y csvs a claude para integrarlos en el reporte pdf final.')

---
## Sección 21 — Discusión

### Comparación con el Estado del Arte

El pipeline clásico propuesto se ubica en la misma categoría que Hoque et al. (2021) [3] y Mustafa et al. (2025) [8], ambos métodos clásicos que reportan IoU de 0.68–0.99 **sobre imágenes de célula individual**. Al evaluar el mismo pipeline bajo condiciones equivalentes (Cx22, célula individual), los resultados son comparables — lo que confirma que el método clásico es una línea base válida.

Frente a los métodos de aprendizaje profundo (U-Net [13], SPCNet [7], Sumon et al. [6]), el pipeline clásico tiene menor IoU (diferencia estimada de 10–20 puntos porcentuales), pero ofrece ventajas concretas en el contexto de este proyecto:

| Criterio | Pipeline clásico (este trabajo) | U-Net / SPCNet |
|----------|:---:|:---:|
| Requiere entrenamiento | No | Sí (~500+ imágenes etiquetadas) |
| Requiere GPU | No (CPU estándar) | Sí |
| Interpretabilidad | Total (7 pasos documentados) | Limitada (caja negra) |
| Velocidad de inferencia | ~12 img/s en CPU | ~50 img/s en GPU / ~1 img/s en CPU |
| Reproducibilidad | Determinista (sin aleatoriedad) | Depende de semilla y hardware |
| Costo de ajuste | 6 hiperparámetros | Millones de parámetros |

Si el pipeline propuesto no supera a los métodos de aprendizaje profundo en IoU, esto se justifica por el diferencial de recursos: un método con cero requisitos de GPU y cero datos de entrenamiento etiquetados que alcanza IoU ≈ 0.60–0.72 en célula individual es una herramienta válida para laboratorios con hardware limitado.

### Efecto del Desajuste de Escala

El resultado más importante de este proyecto es **cuantitativo y metodológico**: el mismo pipeline sin re-ajuste de parámetros obtiene IoU~0.125 en SIPaKMeD (portaobjeto completo, núcleo <1% del área) y un IoU significativamente mayor en Cx22 (célula individual, núcleo ~20–50% del área). Esta diferencia no refleja una falla del método sino una **consecuencia matemática inevitable** del denominador de la IoU:

```
IoU = |P ∩ GT| / |P ∪ GT|
```

Cuando el área del núcleo anotado representa <1% del área total de la imagen, `|P ∪ GT|` crece órdenes de magnitud respecto al caso de célula individual, colapsando la métrica independientemente de qué método se use. Esto coincide con lo documentado por Nateghi et al. (2024) [5] sobre el sesgo de dataset como principal fuente de inconsistencia entre publicaciones.

### Implicaciones para la Literatura

Los resultados de este trabajo tienen una implicación directa: **los IoU reportados en la literatura solo son comparables cuando el formato de imagen es el mismo**. Trabajos que reportan IoU~0.74 sobre célula individual no son directamente comparables con trabajos que evalúan sobre portaobjeto completo. Este análisis ofrece evidencia empírica de esa diferencia, útil para contextualizar futuros reportes.

### Limitaciones

1. La umbralización adaptativa no distingue entre núcleos y artefactos morfológicamente similares (manchas de tinción, detritos celulares).
2. El pipeline no maneja explícitamente núcleos superpuestos en regiones de alta densidad celular.
3. Los parámetros fueron fijados para SIPaKMeD; una búsqueda sistemática sobre Cx22 puede mejorar los resultados en ese dataset.

---
## Sección 23 — Repositorio GitHub

El código fuente, las figuras generadas y los archivos de métricas están disponibles en el repositorio público:

> **Repositorio:** `https://github.com/<usuario>/m3a5-segmentacion-nucleos-cervical`

### Contenido del repositorio

```
m3a5-segmentacion-nucleos-cervical/
├── README.md
├── M3A5_DEL1_Segmentacion_Nucleos_SIPaKMeD_Cx22.ipynb
├── figures/
│   ├── fig1_pipeline_etapas.png
│   ├── fig2_metricas_globales.png
│   ├── fig3_iou_categorias.png
│   ├── fig4_paneles_segmentacion.png
│   ├── fig5_cx22_pipeline.png
│   ├── fig6_cx22_paneles.png
│   ├── fig7_cx22_histograma.png
│   └── fig8_comparativa.png
├── results/
│   ├── metricas_sipakmed.csv
│   ├── metricas_cx22.csv
│   └── comparativa_datasets.csv
└── future_work/
    └── FUTURE_WORK.md
```

### README — Secciones incluidas

1. **Descripción del proyecto** — objetivo, datasets, pipeline de 7 etapas
2. **Requisitos** — Python 3.10+, OpenCV, scikit-image, kagglehub (sin GPU)
3. **Cómo ejecutar** — instrucciones paso a paso para Colab
4. **Resultados principales** — tabla comparativa SIPaKMeD vs Cx22
5. **Trabajo futuro** — enlace a `future_work/FUTURE_WORK.md`
6. **Cómo citar** — referencia al proyecto

El archivo `future_work/FUTURE_WORK.md` documenta las 5 extensiones priorizadas con su análisis costo-beneficio, cumpliendo el requisito explícito de la actividad de incluir trabajo futuro en el repositorio.



---
## Sección 22 — Trabajo Futuro

### Próximos Pasos Técnicos

Las siguientes extensiones están ordenadas por relación costo-beneficio, de mayor a menor prioridad para laboratorios con hardware estándar:

**1. Optimización de hiperparámetros sobre Cx22**
- *Qué:* Búsqueda en cuadrícula sobre `UMBRAL_BLOQUE` ∈ {21, 35, 51}, `UMBRAL_C` ∈ {3, 5, 7} y radios morfológicos ∈ {2, 3, 4, 5} mediante validación cruzada estratificada (k=5) sobre la partición de entrenamiento de Cx22.
- *Costo:* ~2–3 horas CPU, sin GPU, sin datos adicionales.
- *Beneficio esperado:* +5–10% IoU en Cx22.
- *Contra:* Riesgo de sobreajuste al formato de célula individual; parámetros óptimos en Cx22 pueden degradar resultados en SIPaKMeD.

**2. Separación de células superpuestas (watershed)**
- *Qué:* Integrar watershed morfológico sobre el mapa de distancias de la máscara binaria para separar instancias adyacentes. El watershed resuelve el caso de fallo más frecuente: núcleos en contacto que el etiquetado de componentes conectados fusiona.
- *Costo:* ~4 horas de implementación; disponible en `scipy.ndimage` sin GPU.
- *Beneficio:* Mejora directa en IoU de instancias en imágenes con alta densidad celular (categorías Koilocytotic y Dyskeratotic de SIPaKMeD).
- *Contra:* Añade sensibilidad al parámetro de umbral de mínimos locales; puede sobre-separar núcleos grandes displásicos.

**3. Comparación formal contra U-Net ligero (MobileNetV2 encoder)**
- *Qué:* Entrenar una U-Net con encoder MobileNetV2 pre-entrenado en ImageNet sobre el 80% de Cx22; evaluar en el 20% restante usando las mismas 4 métricas.
- *Costo:* ~4 GPU-horas en Google Colab (T4 gratuita).
- *Beneficio:* Cuantifica con rigor el trade-off entre interpretabilidad y precisión; permite afirmar con datos propios si la diferencia de IoU justifica el costo de infraestructura.
- *Contra:* Requiere GPU y ~1 000 imágenes etiquetadas; la comparación es válida solo si el conjunto de entrenamiento y prueba son idénticos.

**4. Extracción de features morfológicas para clasificación Bethesda**
- *Qué:* A partir de las instancias detectadas, extraer por núcleo: área, perímetro, excentricidad, solidez, densidad de cromatina (intensidad media en el canal de grises), y relación núcleo-citoplasma (usando las máscaras de citoplasma de Cx22). Clasificar con SVM o Random Forest en las 5 categorías de Bethesda.
- *Costo:* Bajo (usa detecciones ya generadas; ~2 horas de implementación).
- *Beneficio:* Cierra el pipeline imagen → diagnóstico sin aprendizaje profundo; habilita una aplicación clínica directa del segmentador.
- *Contra:* La calidad de clasificación depende directamente de la calidad de la segmentación; errores de segmentación se propagan a errores de clasificación.

**5. Validación en APACS23**
- *Qué:* Aplicar el mismo pipeline (sin re-ajuste) sobre el dataset APACS23 [11], que contiene ~37 000 células segmentadas a nivel de píxel en imágenes de 1024×1024 px (portaobjeto completo).
- *Costo:* ~1 hora de ejecución; ~5 GB de descarga.
- *Beneficio:* Verifica si la limitación observada en SIPaKMeD (portaobjeto completo, IoU colapsado) se generaliza a APACS23 con diferente protocolo de escaneado.
- *Contra:* Mayor uso de almacenamiento y memoria RAM.

### Reflexión Costo-Beneficio

| Extensión | Costo computacional | Beneficio principal | Riesgo |
|-----------|:-------------------:|---------------------|--------|
| Optimización hiperparam. (Cx22) | ~3h CPU | +5–10% IoU Cx22 | Sobreajuste al dataset |
| Watershed superpuestos | ~4h implementación | Resuelve fusión de instancias | Sensibilidad a parámetros |
| U-Net ligero | ~4h GPU (Colab) | +20–30% IoU, benchmark formal | Pierde interpretabilidad |
| Features Bethesda | ~2h implementación | Pipeline imagen→diagnóstico | Depende de buena segmentación |
| Validación APACS23 | ~1h ejecución | Generalización confirmada | Almacenamiento adicional |

**Conclusión sobre trabajo futuro:** El enfoque clásico presenta la mejor relación costo-beneficio para laboratorios sin acceso a GPU. Si se dispone de GPU y de suficientes imágenes etiquetadas (>500), U-Net ligero es la extensión natural que cierra la brecha de IoU. La extracción de features morfológicas para clasificación Bethesda representa la mayor contribución clínica posible a partir de los resultados actuales, ya que no requiere GPU ni datos adicionales y convierte el segmentador en una herramienta diagnóstica completa.

---## Sección 20 — Notas de cierre para el reporte**Hallazgo principal (a reportar en Discusión):**- En **SIPaKMeD** (portaobjeto completo, núcleo <1% del área) el IoU colapsa por desbalance extremo de clases → confirma la limitación documentada en M3.A4.- En **Cx22** (célula individual, núcleo 20–50% del área) el **mismo pipeline sin re-ajuste** recupera un IoU mucho mayor → **valida que el bajo IoU previo era un artefacto de escala del protocolo de evaluación, no una falla del método**.- La **Precisión de píxel** se mantiene alta en ambos casos, demostrando que es engañosa bajo desbalance — por eso IoU y Dice son las métricas estándar en segmentación biomédica.**Trabajo futuro (reflexión de costo/beneficio):**1. **Búsqueda de hiperparámetros** sobre Cx22 (bloque, C, radios morfológicos) — costo ~2–3 h CPU, beneficio +5–10% IoU, contra: riesgo de sobreajuste.2. **Comparación contra U-Net ligero** — costo ~4 GPU-horas, beneficio: caracterización formal del trade-off precisión vs. interpretabilidad.3. **Extracción de features morfológicas por núcleo** (área, excentricidad, cromatina) para clasificación Bethesda con ML clásico — cierra el pipeline imagen→diagnóstico manteniendo interpretabilidad.**Repositorio:** subir este notebook + figuras + README autoexplicativo a GitHub, incluyendo sección de trabajo futuro (requisito de la actividad).